# Lecture 5 - pandas I

**Lecture Topic**: Broadcasting and the cost of Python loops, then an introduction to pandas: Series, DataFrames, selecting data and index alignment


## Announcements

- We will discuss the project scope and teams on Wednesday
- ClassBuzz poll for today: `M2RA`

## Recap

In the last lecture, we covered the following topics:
 - Scripts vs notebooks
 - Regular expressions, and how to check them
 - NumPy arrays and dtypes (and what happens silently when they don't fit)
 - Indexing, boolean masks and `NaN`
 - Multidimensional arrays and operating along an `axis`

Today, we will cover the following topics:
 - Broadcasting
 - Views vs copies
 - A first look at cost: Python loops vs NumPy
 - pandas: Series, DataFrames and the Index
 - Inspecting a dataset
 - Selecting subsets of the data
 - Index alignment

We'll start by finishing off NumPy from Friday.

## Broadcasting

The fact that NumPy operates on an element-wise basis means that in principle arrays must always match one another's shape. However, NumPy will also helpfully "*broadcast*" dimensions when possible.

In [ ]:
import numpy as np

print(np.arange(3) + 5)                 # a scalar to a 1D array
print(np.ones((3, 3)) + np.arange(3))   # a 1D array to every row of a 2D array

We can also broadcast in two directions at a time:

In [ ]:
a = np.arange(3).reshape((3, 1))
b = np.arange(3)
print(a, '+', b, '=\n', a + b)

Pictorially:

![Illustration of broadcasting](images/fig_broadcast_visual_1.png)

([image source](http://www.astroml.org/book_figures/appendix/fig_broadcast_visual.html))

### Rules of Broadcasting

When operating on two arrays, NumPy compares their shapes element-wise. It starts with the trailing dimensions, and works its way forward. Two dimensions are compatible when

* they are equal, or
* one of them is 1 (or missing); in this case NumPy will 'stretch' it to match the other.

If these conditions are not met, a `ValueError: operands could not be broadcast together` exception is thrown.

Note that all of this happens without ever actually creating the expanded arrays in memory, which can save lots of memory when the arrays in question are large.

### Questions:

What will the result of adding `arr1` with `arr2` be in the following cases?

In [ ]:
arr1 = np.ones((2, 3))
arr2 = np.ones((2, 1))

arr1 + arr2

In [ ]:
arr1 = np.ones((1, 3))
arr2 = np.ones((2, 1))

arr1 + arr2

In [ ]:
arr1 = np.ones((1, 3))
arr2 = np.ones((1, 2))

arr1 + arr2

### Quick quiz

What will be the shape of `C` given the following arrays and operation:

```python
A = np.ones((3, 1, 5))
B = np.ones((1, 4, 1))
C = A + B
```

### Quick quiz: spot the bug

An agent was asked for a function that removes each station's average from an array of temperature readings with shape `(stations, days)`. It wrote this, and tested it:

In [ ]:
def remove_station_mean(data):
    return data - data.mean(axis=1)

test = np.array([[1., 2., 3.],
                 [4., 5., 6.],
                 [7., 8., 9.]])
remove_station_mean(test)

Is it correct?

If it were, every row would now have a mean of zero:

In [ ]:
remove_station_mean(test).mean(axis=1)

`data.mean(axis=1)` has shape `(3,)`, which broadcasts against the *last* dimension, so each **column** had a different station's mean taken off it.

And on real data, with 4 stations and 7 days:

In [ ]:
remove_station_mean(np.ones((4, 7)))

The fix is to keep the dimension we averaged over, so it broadcasts the right way:

In [ ]:
def remove_station_mean(data):
    return data - data.mean(axis=1, keepdims=True)

print(remove_station_mean(test).mean(axis=1))
print(remove_station_mean(np.ones((4, 7))).shape)

The lesson: **test with arrays that have a different length in every dimension**. A square array can't tell rows from columns.

## Views, not Copies

Note that reshaping and slicing (like most NumPy operations), wherever possible, provide a **view** of *the same memory*:

In [ ]:
arr = np.arange(8)
arr_view = arr.reshape(2, 4)
arr_view[0, 0] = 1000
print(arr)

This lack of copying allows for very efficient vectorized operations, but this power should be used carefully - if used badly it can lead to some bugs that are hard to track down.

If in doubt, you can always copy the data to a different block of memory with the **`copy()`** method.

## A first look at cost

Let's compute the tip as a percentage of the fare for a million taxi trips, first with a Python loop, then with NumPy.

In [ ]:
rng = np.random.default_rng(seed=0)
fares = rng.uniform(5, 60, size=1_000_000)
tips = rng.uniform(0, 10, size=1_000_000)

def tip_pct_loop(fares, tips):
    out = []
    for tip, fare in zip(tips, fares):
        out.append(tip / fare * 100)
    return out

def tip_pct_numpy(fares, tips):
    return tips / fares * 100

### Quick quiz

How much faster do you think the NumPy version will be?

 - A. About the same
 - B. 2-5x faster
 - C. 10-100x faster
 - D. More than 1000x faster

In [ ]:
fares_list, tips_list = fares.tolist(), tips.tolist()

%timeit tip_pct_loop(fares_list, tips_list)
%timeit tip_pct_numpy(fares, tips)

In [ ]:
np.allclose(tip_pct_loop(fares_list, tips_list), tip_pct_numpy(fares, tips))

### Why?

Each time around a Python loop, the interpreter has to look up what `tips[i]` is, check its type, work out what `/` means for that type, and create a new Python object for the result.

NumPy does that once for the whole array, then runs a tight loop in compiled C over one contiguous block of memory.

The memory is different too. A NumPy array stores just the numbers, whereas a list stores a pointer to a separate Python object for each one:

In [ ]:
import sys

print(f"array: {fares.nbytes / 1e6:.0f} MB")
print(f"list:  {(sys.getsizeof(fares_list) + sum(sys.getsizeof(x) for x in fares_list)) / 1e6:.0f} MB")

Both versions give the same answer, and on 10,000 rows you would never notice the difference. At 100 million rows you would.

This is exactly the kind of choice an agent makes for you without mentioning it. We'll come back to it properly in Module C, where we'll measure how the cost grows with the size of the data.

## Introduction to pandas

pandas is a powerful library especially designed to work with tabular data, and it has powerful time-series analysis tools.

These slides lean heavily on a fantastic [Pandas Workshop](https://github.com/stefmolin/pandas-workshop/tree/main) put together by Stefanie Molin.

pandas is built on top of NumPy, so a lot of what we've just seen (dtypes, boolean masks, vectorized operations) carries straight over.

We will begin by introducing the `Series`, `DataFrame`, and `Index` classes, which are the basic building blocks of the pandas library, and showing how to work with them.

In [ ]:
import pandas as pd

## Anatomy of a DataFrame

A **DataFrame** is composed of one or more **Series**. The names of the **Series** form the column names, and the row labels form the **Index**.

In [ ]:
taxis = pd.read_csv('../data/2019_Yellow_Taxi_Trip_Data.csv', nrows=5)
taxis

*Source: [NYC Open Data](https://data.cityofnewyork.us/Transportation/2019-Yellow-Taxi-Trip-Data/2upf-qytp), a sample of 10,000 trips from October 2019.*

#### Series:

In [ ]:
taxis.fare_amount

#### Columns:

In [ ]:
taxis.columns

#### Index:

In [ ]:
taxis.index 

## Creating DataFrames

We can create DataFrames from a variety of sources such as other Python objects, flat files, webscraping, and API requests. Here, we will see just a couple of examples, but be sure to check out [this page](https://pandas.pydata.org/docs/user_guide/io.html) in the documentation for a complete list.

### From scratch

A common approach is using a dictionary as the argument to `pd.DataFrame()`:

In [ ]:
data = {
    'apples': [3, 2, 0, 1],
    'oranges': [0, 3, 7, 2]
}

pd.DataFrame(data)

We can also specify an index:

In [ ]:
pd.DataFrame(data, index=['Ke', 'Julian', 'Duong', 'Andreas'])

### Using a flat file

In [ ]:
taxis = pd.read_csv('../data/2019_Yellow_Taxi_Trip_Data.csv')

*Tip: `df.to_csv('data.csv')` writes a DataFrame to a new file called `data.csv`.*

## Inspecting the data

Now that we have some data, we need to perform an initial inspection of it. This gives us information on what the data looks like, how many rows/columns there are, and how much data we have.

#### How many rows and columns are there?

In [ ]:
taxis.shape

#### What type of data does each column currently hold?

In [ ]:
taxis.dtypes

Notice that the pickup and dropoff times are stored as strings (`str`), not dates. pandas doesn't guess, so it's up to us to convert them. We'll do that on Wednesday.

#### What does the data look like?

In [ ]:
taxis.head()

Sometimes there may be extraneous data at the end of the file, so checking the bottom few rows is also important:

In [ ]:
taxis.tail()

#### Get some summary statistics

In [ ]:
taxis[['passenger_count', 'trip_distance', 'fare_amount', 'tip_amount']].describe()

Question: does anything here look odd?

Some of the fares are negative (these are mostly refunds and disputes). It's worth knowing about them before you compute an average fare, and an agent summarising this data won't necessarily mention them.

## Extracting subsets

A crucial part of working with DataFrames is extracting subsets of the data: finding rows that meet a certain set of criteria, isolating columns/rows of interest, etc. After narrowing down our data, we are closer to discovering insights. This section will be the backbone of many analysis tasks.

#### Selecting columns

We can select columns as attributes if their names would be valid Python variables:

In [ ]:
taxis.fare_amount

Otherwise (or if you prefer) we select them as keys. We can also select multiple columns at once this way, using a list:

In [ ]:
taxis[['fare_amount', 'tip_amount']]

#### Selecting rows

In [ ]:
taxis[100:104]

#### Indexing

We use `iloc[]` to select rows and columns by their position:

In [ ]:
taxis.iloc[100:104, [0, 10, 13]]

We use `loc[]` to select by name:

In [ ]:
taxis.loc[100:104, 'fare_amount':'tip_amount']

### Quick quiz

How many rows does `taxis.loc[100:104]` return? And `taxis.iloc[100:104]`?

 - A. 4 and 4
 - B. 5 and 4
 - C. 4 and 5
 - D. 5 and 5

In [ ]:
print(len(taxis.loc[100:104]))
print(len(taxis.iloc[100:104]))

`loc` slices by label and includes the end label. `iloc` slices by position, and (like everything else in Python) doesn't include the end.

#### Filtering with Boolean masks

A **Boolean mask** is an array-like structure of Boolean values. It's a way to specify which rows/columns we want to select (`True`) and which we don't (`False`), just like with NumPy.

Here's an example of a Boolean mask for trips with a fare of more than $50 that were paid by card (`payment_type` 1):

In [ ]:
(taxis.fare_amount > 50) & (taxis.payment_type == 1)

**Important**: Take note of the syntax here. We surround each condition with parentheses, and we use bitwise operators (`&`, `|`, `~`) instead of logical operators (`and`, `or`, `not`).

If we use `and` instead, we get an error:

In [ ]:
taxis[taxis.fare_amount > 50 and taxis.payment_type == 1]

`and` needs a single `True` or `False`, and pandas won't guess whether you mean "all of them" or "any of them".

We can use the mask to select the subset of rows:

In [ ]:
taxis[(taxis.fare_amount > 50) & (taxis.payment_type == 1)]

*Tip: Boolean masks can be used with `loc[]` too, e.g. `taxis.loc[mask, 'tip_amount']`.*

An alternative to this is the `query()` method:

In [ ]:
taxis.query("fare_amount > 50 and payment_type == 1")

*Tip: Here, we can use both logical operators and bitwise operators.*

## Index alignment

This is the biggest difference between pandas and NumPy. When you do arithmetic on two Series, pandas lines them up by their index **labels**, not by their position.

In [ ]:
fares = pd.Series([10.0, 20.0, 30.0], index=['a', 'b', 'c'])
tips = pd.Series([3.0, 2.0, 1.0], index=['c', 'b', 'a'])
tips / fares

This is usually what you want: rows match up even if one of them has been sorted differently.

### Quick quiz

What is `s1 + s2`?

 - A. `[11, 22, 33]`
 - B. `[NaN, 12, 23, NaN]`
 - C. `[12, 23]`
 - D. An error

In [ ]:
s1 = pd.Series([1, 2, 3], index=[0, 1, 2])
s2 = pd.Series([10, 20, 30], index=[1, 2, 3])
s1 + s2

The result has every label from either side. Where a label only appears in one of them, you get `NaN`, with no error and no warning. Notice the dtype has changed to `float64` too, since an integer can't be `NaN`.

## Summary

Today, we covered the following topics:
 - Broadcasting (and why you should test with arrays of different lengths)
 - Views vs copies
 - Python loops vs NumPy: time and memory
 - pandas Series, DataFrames and the Index
 - Inspecting a dataset
 - Selecting data with `[]`, `loc`, `iloc`, Boolean masks and `query()`
 - Index alignment

On Wednesday, the course project, then pandas II: cleaning, group-by and merging, and more of the ways pandas can go silently wrong.